# this is training the CNNPZ model on the noisy mock data with randomly dropping bands

In [ ]:
import sys

from packaging import version
import sklearn
from sklearn.model_selection import KFold, train_test_split

assert version.parse(sklearn.__version__) >= version.parse("1.0.1")

import tensorflow as tf

assert version.parse(tf.__version__) >= version.parse("2.8.0")

#import tensorflow_probability as tfp

import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

import pandas as pd
import h5py
import numpy as np

import json
import os

from matplotlib import colors

In [ ]:
import cnnpz

In [ ]:
# Parametric paths (edit via environment variables, or defaults below)
USER = os.environ.get("USER", "jaimerz")
PSCRATCH = os.environ.get("PSCRATCH", f"/pscratch/sd/{USER[0]}/{USER}")
PROJECT_HOME = os.environ.get("CNNPZ_PROJECT_HOME", f"/global/homes/{USER[0]}/{USER}/UCL")

CARDINAL_DATA_ROOT = os.path.join(PSCRATCH, "cnnpz", "cardinal") + "/"
MODEL_ROOT = os.path.join(PSCRATCH, "cnnpz", "noisy_Cardinal", "models")
PRETRAINING_DATA_ROOT = os.path.join(PSCRATCH, "pop-cosmos-data")
FILTER_ROOT = os.path.join(PROJECT_HOME, "rail_base", "src", "rail", "examples_data", "estimation_data", "data", "FILTER") + "/"


## Load training and test data, filter curves

In [ ]:
saveroot = CARDINAL_DATA_ROOT
fname = saveroot + "train_100k_noisy_y1_i23.parquet"
training_y1 = pd.read_parquet(fname)
training_y1 = training_y1.rename(columns={'Roman_obs_Y106': 'mag_Y_roman',
                                              'Roman_obs_J129': 'mag_J_roman', 
                                              'Roman_obs_H158': 'mag_H_roman'})

fname = saveroot + "train_100k_noisy_y10_i25.4.parquet"
training_y10 = pd.read_parquet(fname)
training_y10 = training_y10.rename(columns={'Roman_obs_Y106': 'mag_Y_roman',
                                              'Roman_obs_J129': 'mag_J_roman', 
                                              'Roman_obs_H158': 'mag_H_roman'})

saveroot = CARDINAL_DATA_ROOT
fname = saveroot + "test_100k_noisy_y1_i23.parquet"
test_y1 = pd.read_parquet(fname)
test_y1 = test_y1.rename(columns={'Roman_obs_Y106': 'mag_Y_roman',
                                              'Roman_obs_J129': 'mag_J_roman', 
                                              'Roman_obs_H158': 'mag_H_roman'})

fname = saveroot + "test_100k_noisy_y10_i25.4.parquet"
test_y10 = pd.read_parquet(fname)
test_y10 = test_y10.rename(columns={'Roman_obs_Y106': 'mag_Y_roman',
                                              'Roman_obs_J129': 'mag_J_roman', 
                                              'Roman_obs_H158': 'mag_H_roman'})

### Transform data, make X_train, Y_train, and test dataset, variants of this dataset

In [ ]:
# get the LSST and roman filter curves:
filter_root = FILTER_ROOT

wave = {
    "Y":106,
    "J":129,
    "H":158,
}
lsst_filter_curves = {}
roman_filter_curves = {}
for b in "ugrizy":
  lsst_filter_curves[b] = np.loadtxt(filter_root + f'DC2LSST_{b}.res')

for b in "YJH":
  roman_filter_curves[b] = np.loadtxt(filter_root + f'roman_{b}{wave[b]}.res')

lambda_min = lsst_filter_curves['u'][:,0].min()
lambda_max = roman_filter_curves['H'][:,0].max()
print(lambda_min, lambda_max)

In [ ]:
# build the filter bank (avoids double-counting overlapping filters) and bin it down to
# the CNN's 32 input bins -- 9 bands now, bringing back Roman Y106 alongside ugrizy+J+H
lambda_common = np.linspace(lambda_min, lambda_max, 1000)
bands = "ugrizyYJH"
filter_curves = {**lsst_filter_curves, **roman_filter_curves}
filters_array = cnnpz.interpolate_filter_curves(filter_curves, lambda_common)

n_lambda_bins = 32
lambda_bin_centers, _ = cnnpz.make_lambda_bins(lambda_common, n_lambda_bins)
filters_binned = cnnpz.bin_filters(filters_array, n_lambda_bins, lambda_common)
print("filters_array:", filters_array.shape, "filters_binned:", filters_binned.shape)

# cnnpz's data-prep functions take plain (n_filters, n_sources) magnitude arrays instead
# of a dataframe -- this bridges our per-project column naming to that.
roman_bands = "YJH"
def mags_from_df(df, bands):
    cols = [f"mag_{b}_roman" if b in roman_bands else f"mag_{b}_lsst" for b in bands]
    return df[cols].to_numpy().T  # (n_filters, n_sources)

i_idx = bands.index("i")
nir_idx = np.array([bands.index(b) for b in "JH"])

In [ ]:
mags_train_y1 = mags_from_df(training_y1, bands)
mags_test_y1 = mags_from_df(test_y1, bands)
mags_train_y10 = mags_from_df(training_y10, bands)
mags_test_y10 = mags_from_df(test_y10, bands)

X_y1, Y_y1 = cnnpz.transform_data_to_XY(mags_train_y1, training_y1["redshift"].to_numpy(), filters_array, n_lambda_bins, lambda_common, mag_i=mags_train_y1[i_idx], apply_stretch = False)
X_test_y1, Y_test_y1 = cnnpz.transform_data_to_XY(mags_test_y1, test_y1["redshift"].to_numpy(), filters_array, n_lambda_bins, lambda_common, mag_i=mags_test_y1[i_idx], apply_stretch = False)

X_y10, Y_y10 = cnnpz.transform_data_to_XY(mags_train_y10, training_y10["redshift"].to_numpy(), filters_array, n_lambda_bins, lambda_common, mag_i=mags_train_y10[i_idx], apply_stretch = False)
X_test_y10, Y_test_y10 = cnnpz.transform_data_to_XY(mags_test_y10, test_y10["redshift"].to_numpy(), filters_array, n_lambda_bins, lambda_common, mag_i=mags_test_y10[i_idx], apply_stretch = False)

## Making datasets with missing bands - NIR removal

In [ ]:
# let's construct a dataset where half of the sample do not have NIR measurements:
X_y1_misnir, Y_y1_misnir = cnnpz.make_incomplete_nir_data(mags_train_y1, training_y1["redshift"].to_numpy(), filters_array, n_lambda_bins, lambda_common, nir_idx, mag_i=mags_train_y1[i_idx], apply_stretch = False)
X_test_y1_misnir, Y_test_y1_misnir = cnnpz.make_incomplete_nir_data(mags_test_y1, test_y1["redshift"].to_numpy(), filters_array, n_lambda_bins, lambda_common, nir_idx, mag_i=mags_test_y1[i_idx], apply_stretch = False)

X_y10_misnir, Y_y10_misnir = cnnpz.make_incomplete_nir_data(mags_train_y10, training_y10["redshift"].to_numpy(), filters_array, n_lambda_bins, lambda_common, nir_idx, mag_i=mags_train_y10[i_idx], apply_stretch = False)
X_test_y10_misnir, Y_test_y10_misnir = cnnpz.make_incomplete_nir_data(mags_test_y10, test_y10["redshift"].to_numpy(), filters_array, n_lambda_bins, lambda_common, nir_idx, mag_i=mags_test_y10[i_idx], apply_stretch = False)

# visualize the data

In [ ]:
cnnpz.visualize_the_data(X_y1, Y_y1, lambda_bin_centers, title="Y1 training example")

In [ ]:
cnnpz.visualize_the_data(X_y1_misnir, Y_y1_misnir, lambda_bin_centers, title="Y1 misnir example")

In [ ]:
cnnpz.visualize_the_data(X_y10, Y_y10, lambda_bin_centers, title="Y10 training example")

In [ ]:
cnnpz.visualize_the_data(X_y10_misnir, Y_y10_misnir, lambda_bin_centers, title="Y10 misnir example")

# training ensemble model

In [ ]:
# train on Y1 complete (or load if already trained)
from cnnpz import build_model
root = MODEL_ROOT
save_dir = root + "/y1_complete_curve_9band_ensemble_CNN_6layers_noshape"
if os.path.exists(os.path.join(save_dir, "norm_params.json")):
    trained_models = cnnpz.load_ensemble(save_dir=save_dir)
    histories = None
else:
    trained_models, histories = cnnpz.train_ensembles(build_model, X_y1, Y_y1)
    cnnpz.save_ensemble(trained_models, save_dir=save_dir)

In [ ]:
if histories is not None:
    cnnpz.plot_ensemble_losses(histories, ylim=(0, 0.001))

In [ ]:
# Final prediction on test set
y_pred_ensemble, y_pred_STD = cnnpz.ensemble_predict(trained_models, X_test_y1)

In [ ]:
redshift_bins = np.linspace(0,2.5,11)
imag_bins = np.linspace(18, 25.5,11)
stats2, redshift_stats2, imag_stats2 = cnnpz.get_all_stats(Y_test_y1, y_pred_ensemble.flatten(), test_y1['mag_i_lsst'], save=True, saveroot=save_dir + "/stats.pkl", 
                                                     redshift_bins = redshift_bins, imag_bins = imag_bins)

cnnpz.plot_stats(stats2, redshift_stats2, imag_stats2, Y_test_y1, y_pred_ensemble.flatten(), 
           redshift_bins, imag_bins, test_y1['mag_i_lsst'], save_path=save_dir + '/photoz_stats.png')

In [ ]:
# also plot the STD as a function of redshift and i-band magnitude:
fig,axarr=plt.subplots(1,3,figsize=[13,4])
plt.sca(axarr[0])
ind = y_pred_STD.flatten() >= 0.05
plt.scatter(Y_test_y1[~ind], y_pred_ensemble.flatten()[~ind], s=0.1, color='k')
plt.scatter(Y_test_y1[ind], y_pred_ensemble.flatten()[ind], s=0.5, color='r', label="STD_y > 0.05")
plt.xlabel("Truth redshift")
plt.ylabel("predicted redshift (mean)")
plt.legend()

plt.sca(axarr[1])
plt.scatter(Y_test_y1, y_pred_STD.flatten(), s=0.1, color='k')
plt.xlabel("Truth redshift")
plt.ylabel("predicted redshift (STD)")

plt.sca(axarr[2])
plt.scatter(test_y1['mag_i_lsst'], y_pred_STD.flatten(), s=0.1, color='k')
plt.xlabel("i magnitude")
plt.ylabel("predicted redshift (STD)")

plt.tight_layout()

In [ ]:
# now let's train on the data with dropped bands (or load if already trained)
root = MODEL_ROOT
save_dir = root + "/y1_misnir_curve_9band_ensemble_CNN_6layers_noshape"
if os.path.exists(os.path.join(save_dir, "norm_params.json")):
    trained_models = cnnpz.load_ensemble(save_dir=save_dir)
    histories = None
else:
    trained_models, histories = cnnpz.train_ensembles(build_model, X_y1_misnir, Y_y1_misnir)
    cnnpz.save_ensemble(trained_models, save_dir=save_dir)

In [ ]:
y_pred_ensemble_misnir, y_pred_STD_misnir = cnnpz.ensemble_predict(trained_models, X_test_y1_misnir)

In [ ]:
redshift_bins = np.linspace(0,2.5,11)
imag_bins = np.linspace(18, 25.5,11)
stats2, redshift_stats2, imag_stats2 = cnnpz.get_all_stats(Y_test_y1_misnir, y_pred_ensemble_misnir.flatten(), test_y1['mag_i_lsst'], save=True, saveroot=save_dir + "/stats.pkl", 
                                                     redshift_bins = redshift_bins, imag_bins = imag_bins)

cnnpz.plot_stats(stats2, redshift_stats2, imag_stats2, Y_test_y1_misnir, y_pred_ensemble_misnir.flatten(), 
           redshift_bins, imag_bins, test_y1['mag_i_lsst'], save_path=save_dir + '/photoz_stats.png')

In [ ]:
# let's apply the complete test data on the model trained with incomplete data
y_pred_ensemble2, y_pred_STD2= cnnpz.ensemble_predict(trained_models, X_test_y1)

In [ ]:
redshift_bins = np.linspace(0,2.5,11)
imag_bins = np.linspace(18, 25.5,11)
stats2, redshift_stats2, imag_stats2 = cnnpz.get_all_stats(Y_test_y1, y_pred_ensemble2.flatten(), test_y1['mag_i_lsst'], save=True, saveroot=save_dir + "/stats_on_complete_test.pkl", 
                                                     redshift_bins = redshift_bins, imag_bins = imag_bins)

cnnpz.plot_stats(stats2, redshift_stats2, imag_stats2, Y_test_y1, y_pred_ensemble2.flatten(), 
           redshift_bins, imag_bins, test_y1['mag_i_lsst'], save_path=save_dir + '/photoz_stats.png')

## Cardinal Y10

In [ ]:
# train on Y10 complete (or load if already trained)
root = MODEL_ROOT
save_dir = root + "/y10_complete_curve_9band_ensemble_CNN_6layers_noshape"
if os.path.exists(os.path.join(save_dir, "norm_params.json")):
    trained_models = cnnpz.load_ensemble(save_dir=save_dir)
    histories = None
else:
    trained_models, histories = cnnpz.train_ensembles(build_model, X_y10, Y_y10)
    cnnpz.save_ensemble(trained_models, save_dir=save_dir)

In [ ]:
if histories is not None:
    cnnpz.plot_ensemble_losses(histories, ylim=(0, 0.005))

In [ ]:
y_pred_ensemble, y_pred_STD = cnnpz.ensemble_predict(trained_models, X_test_y10)

In [ ]:
redshift_bins = np.linspace(0,2.5,11)
imag_bins = np.linspace(18, 25.5,11)
stats2, redshift_stats2, imag_stats2 = cnnpz.get_all_stats(Y_test_y10, y_pred_ensemble.flatten(), test_y10['mag_i_lsst'], save=True, saveroot=save_dir + "/stats.pkl", 
                                                     redshift_bins = redshift_bins, imag_bins = imag_bins)

cnnpz.plot_stats(stats2, redshift_stats2, imag_stats2, Y_test_y10, y_pred_ensemble.flatten(), 
           redshift_bins, imag_bins, test_y10['mag_i_lsst'], save_path=save_dir + '/photoz_stats.png')

In [ ]:
# also plot the STD as a function of redshift and i-band magnitude:
fig,axarr=plt.subplots(1,3,figsize=[13,4])
plt.sca(axarr[0])
ind = y_pred_STD.flatten() >= 0.05
plt.scatter(Y_test_y10[~ind], y_pred_ensemble.flatten()[~ind], s=0.1, color='k')
plt.scatter(Y_test_y10[ind], y_pred_ensemble.flatten()[ind], s=0.5, color='r', label="STD_y > 0.05")
plt.xlabel("Truth redshift")
plt.ylabel("predicted redshift (mean)")
plt.legend()

plt.sca(axarr[1])
plt.scatter(Y_test_y10, y_pred_STD.flatten(), s=0.1, color='k')
plt.xlabel("Truth redshift")
plt.ylabel("predicted redshift (STD)")

plt.sca(axarr[2])
plt.scatter(test_y10['mag_i_lsst'], y_pred_STD.flatten(), s=0.1, color='k')
plt.xlabel("i magnitude")
plt.ylabel("predicted redshift (STD)")

plt.tight_layout()

In [ ]:
# try applying this model to the specsel data: is this consistent?
root = MODEL_ROOT
save_dir = root + "/y10_complete_curve_9band_ensemble_CNN_6layers_noshape"
trained_models = cnnpz.load_ensemble(save_dir=save_dir)

In [ ]:
saveroot = CARDINAL_DATA_ROOT
training_y10_spec = pd.read_parquet(saveroot + "train_100k_noisy_y10_i25.4.SpecSelect.parquet")
# change the name of the roman columns:
training_y10_spec = training_y10_spec.rename(columns={'Roman_obs_Y106': 'mag_Y_roman',
                                              'Roman_obs_J129': 'mag_J_roman', 
                                              'Roman_obs_H158': 'mag_H_roman'})

mags_y10_spec = mags_from_df(training_y10_spec, bands)
X_y10_spec, Y_y10_spec = cnnpz.transform_data_to_XY(mags_y10_spec, training_y10_spec["redshift"].to_numpy(), filters_array, n_lambda_bins, lambda_common, mag_i=mags_y10_spec[i_idx], apply_stretch=False)
y_pred_ensemble_spec, y_pred_STD_spec = cnnpz.ensemble_predict(trained_models, X_y10_spec)

In [ ]:
redshift_bins = np.linspace(0,2.5,11)
imag_bins = np.linspace(18, 25.5,11)
stats2, redshift_stats2, imag_stats2 = cnnpz.get_all_stats(Y_y10_spec, y_pred_ensemble_spec.flatten(), 
                                                     training_y10_spec['mag_i_lsst'], save=True, saveroot=save_dir + "/stats_on_specsel.pkl", 
                                                     redshift_bins = redshift_bins, imag_bins = imag_bins)

cnnpz.plot_stats(stats2, redshift_stats2, imag_stats2, Y_y10_spec, y_pred_ensemble_spec.flatten(), 
           redshift_bins, imag_bins, training_y10_spec['mag_i_lsst'], save_path=save_dir + '/photoz_stats.png')

This works well on the spec selected sample!

## dropping nir data

In [ ]:
# train on Y10 with dropped NIR bands (or load if already trained)
root = MODEL_ROOT
save_dir = root + "/y10_misnir_curve_9band_ensemble_CNN_6layers_noshape"
if os.path.exists(os.path.join(save_dir, "norm_params.json")):
    trained_models = cnnpz.load_ensemble(save_dir=save_dir)
    histories = None
else:
    trained_models, histories = cnnpz.train_ensembles(build_model, X_y10_misnir, Y_y10_misnir)
    cnnpz.save_ensemble(trained_models, save_dir=save_dir)

In [ ]:
if histories is not None:
    cnnpz.plot_ensemble_losses(histories, ylim=(0, 0.01))

In [ ]:
y_pred_ensemble_misnir, y_pred_STD_misnir = cnnpz.ensemble_predict(trained_models, X_test_y10_misnir)

In [ ]:
redshift_bins = np.linspace(0,2.5,11)
imag_bins = np.linspace(18, 25.5,11)
stats2, redshift_stats2, imag_stats2 = cnnpz.get_all_stats(Y_test_y10_misnir, y_pred_ensemble_misnir.flatten(), test_y10['mag_i_lsst'], save=True, saveroot=save_dir + "/stats.pkl", 
                                                     redshift_bins = redshift_bins, imag_bins = imag_bins)

cnnpz.plot_stats(stats2, redshift_stats2, imag_stats2, Y_test_y10_misnir, y_pred_ensemble_misnir.flatten(), 
           redshift_bins, imag_bins, test_y10['mag_i_lsst'], save_path=save_dir + '/photoz_stats.png')

## Spec select sample (no pre-training):

In [ ]:
saveroot = CARDINAL_DATA_ROOT
training_y10_spec = pd.read_parquet(saveroot + "train_100k_noisy_y10_i25.4.SpecSelect.parquet")
# change the name of the roman columns:
training_y10_spec = training_y10_spec.rename(columns={'Roman_obs_Y106': 'mag_Y_roman',
                                              'Roman_obs_J129': 'mag_J_roman', 
                                              'Roman_obs_H158': 'mag_H_roman'})
training_y10_spec = training_y10_spec.reset_index(drop=True) 
mags_y10_spec = mags_from_df(training_y10_spec, bands)
X_y10_spec, Y_y10_spec = cnnpz.transform_data_to_XY(mags_y10_spec, training_y10_spec["redshift"].to_numpy(), filters_array, n_lambda_bins, lambda_common, mag_i=mags_y10_spec[i_idx], apply_stretch=False)


saveroot = CARDINAL_DATA_ROOT
training_y1_spec = pd.read_parquet(saveroot + "train_100k_noisy_y1_i23.SpecSelect.parquet")
# change the name of the roman columns:
training_y1_spec = training_y1_spec.rename(columns={'Roman_obs_Y106': 'mag_Y_roman',
                                              'Roman_obs_J129': 'mag_J_roman', 
                                              'Roman_obs_H158': 'mag_H_roman'})
training_y1_spec = training_y1_spec.reset_index(drop=True) 
mags_y1_spec = mags_from_df(training_y1_spec, bands)
X_y1_spec, Y_y1_spec = cnnpz.transform_data_to_XY(mags_y1_spec, training_y1_spec["redshift"].to_numpy(), filters_array, n_lambda_bins, lambda_common, mag_i=mags_y1_spec[i_idx], apply_stretch=False)

In [ ]:
cnnpz.visualize_the_data(X_y1_spec, Y_y1_spec, lambda_bin_centers, title="Y1 spec example")

In [ ]:
# train on Y1 spec-select (or load if already trained)
root = MODEL_ROOT
save_dir = root + "/y1_specsel_curve_9band_ensemble_CNN_6layers_noshape"
if os.path.exists(os.path.join(save_dir, "norm_params.json")):
    trained_models = cnnpz.load_ensemble(save_dir=save_dir)
    histories = None
else:
    trained_models, histories = cnnpz.train_ensembles(build_model, X_y1_spec, Y_y1_spec)
    cnnpz.save_ensemble(trained_models, save_dir=save_dir)

In [ ]:
if histories is not None:
    cnnpz.plot_ensemble_losses(histories, ylim=(0, 0.002))

In [ ]:
y_pred_ensemble, y_pred_STD = cnnpz.ensemble_predict(trained_models, X_test_y1)

In [ ]:
redshift_bins = np.linspace(0,2.5,11)
imag_bins = np.linspace(18, 25.5,11)
stats2, redshift_stats2, imag_stats2 = cnnpz.get_all_stats(Y_test_y1, y_pred_ensemble.flatten(), test_y1['mag_i_lsst'], save=True, saveroot=save_dir + "/stats.pkl", 
                                                     redshift_bins = redshift_bins, imag_bins = imag_bins)

cnnpz.plot_stats(stats2, redshift_stats2, imag_stats2, Y_test_y1, y_pred_ensemble.flatten(), 
           redshift_bins, imag_bins, test_y1['mag_i_lsst'], save_path=save_dir + '/photoz_stats.png')

### Y10

In [ ]:
# train on Y10 spec-select (or load if already trained)
root = MODEL_ROOT
save_dir = root + "/y10_specsel_curve_9band_ensemble_CNN_6layers_noshape"
if os.path.exists(os.path.join(save_dir, "norm_params.json")):
    trained_models = cnnpz.load_ensemble(save_dir=save_dir)
    histories = None
else:
    trained_models, histories = cnnpz.train_ensembles(build_model, X_y10_spec, Y_y10_spec)
    cnnpz.save_ensemble(trained_models, save_dir=save_dir)

In [ ]:
if histories is not None:
    cnnpz.plot_ensemble_losses(histories, ylim=(0, 0.002))

In [ ]:
y_pred_ensemble, y_pred_STD = cnnpz.ensemble_predict(trained_models, X_test_y10)

In [ ]:
redshift_bins = np.linspace(0,2.5,11)
imag_bins = np.linspace(18, 25.5,11)
stats2, redshift_stats2, imag_stats2 = cnnpz.get_all_stats(Y_test_y10, y_pred_ensemble.flatten(), test_y10['mag_i_lsst'], save=True, saveroot=save_dir + "/stats.pkl", 
                                                     redshift_bins = redshift_bins, imag_bins = imag_bins)

cnnpz.plot_stats(stats2, redshift_stats2, imag_stats2, Y_test_y10, y_pred_ensemble.flatten(), 
           redshift_bins, imag_bins, test_y10['mag_i_lsst'], save_path=save_dir + '/photoz_stats.png')

## Now include pre-training with the pop-cosmos data

In [ ]:
# load pre-training data
fname = os.path.join(PRETRAINING_DATA_ROOT, "mock_catalog_Ch1_26.degrade_lsst_y1.parquet")
pretraining_y1 = pd.read_parquet(fname)
# change the name of the roman columns:
pretraining_y1 = pretraining_y1.rename(columns={'Roman_obs_Y106': 'mag_Y_roman',
                                              'Roman_obs_J129': 'mag_J_roman', 
                                              'Roman_obs_H158': 'mag_H_roman'})

fname = os.path.join(PRETRAINING_DATA_ROOT, "mock_catalog_Ch1_26.degrade_lsst_y10.parquet")
pretraining_y10 = pd.read_parquet(fname)
# change the name of the roman columns:
pretraining_y10 = pretraining_y10.rename(columns={'Roman_obs_Y106': 'mag_Y_roman',
                                              'Roman_obs_J129': 'mag_J_roman', 
                                              'Roman_obs_H158': 'mag_H_roman'})

In [ ]:
# plot comparison in colour-redshift space:
fig,axarr=plt.subplots(1,3,figsize=[10,3],sharey=True)

for ii, colours in enumerate(['gr','ri','iz']):
    plt.sca(axarr[ii])
    ri = pretraining_y10[f'mag_{colours[0]}_lsst'] - pretraining_y10[f'mag_{colours[1]}_lsst']
    red = pretraining_y10['redshift']
    ind = pretraining_y10['mag_i_lsst'] < 25.4
    plt.scatter(red[ind][::50], ri[ind][::50], s=0.1, label="pre-training data")
    
    ri = training_y10[f'mag_{colours[0]}_lsst'] - training_y10[f'mag_{colours[1]}_lsst']
    red = training_y10['redshift']
    plt.scatter(red[::10], ri[::10], s=0.1, label="test data")
    
    plt.grid()
    plt.legend()
    plt.xlabel("redshift")
    plt.ylabel(f"{colours[0]}-{colours[1]}")
    plt.ylim([-1.5,5])

In [ ]:
# pre-training data:
pretraining_y10 = pretraining_y10[pretraining_y10['redshift']<2.4]
pretraining_y10 = pretraining_y10.sample(frac=0.5)
pretraining_y10 = pretraining_y10.reset_index(drop=True)
mags_y10_pre = mags_from_df(pretraining_y10, bands)
X_y10_pre, Y_y10_pre = cnnpz.transform_data_to_XY(mags_y10_pre, pretraining_y10["redshift"].to_numpy(), filters_array, n_lambda_bins, lambda_common, mag_i=mags_y10_pre[i_idx], apply_stretch=False)

In [ ]:
Y_y10_pre.dtype, Y_y10_spec.dtype

In [ ]:
len(Y_y10_pre)

In [ ]:
cnnpz.visualize_the_data(X_y10_pre, Y_y10_pre, lambda_bin_centers, title="Y10 pre-training example")

In [ ]:
# train on Y10 pre-training data (or load if already trained)
root = "/pscratch/sd/q/qhang/cnnpz/noisy_Cardinal/models"
save_dir = root + "/y10_pretrain_curve_9band_ensemble_CNN_6layers_noshape"
if os.path.exists(os.path.join(save_dir, "norm_params.json")):
    trained_models = cnnpz.load_ensemble(save_dir=save_dir)
    histories = None
else:
    trained_models, histories = cnnpz.train_ensembles(build_model, X_y10_pre, Y_y10_pre)
    cnnpz.save_ensemble(trained_models, save_dir=save_dir)

In [ ]:
if histories is not None:
    cnnpz.plot_ensemble_losses(histories, ylim=(0, 0.005))

In [ ]:
# check pre-training performance
y_pred_ensemble, y_pred_STD = cnnpz.ensemble_predict(trained_models, X_y10_pre)

In [ ]:
redshift_bins = np.linspace(0,2.5,11)
imag_bins = np.linspace(18, 25.5,11)
stats2, redshift_stats2, imag_stats2 = cnnpz.get_all_stats(Y_y10_pre, y_pred_ensemble.flatten(), 
                                                     pretraining_y10['mag_i_lsst'], save=True, saveroot=save_dir + "/stats.pkl", 
                                                     redshift_bins = redshift_bins, imag_bins = imag_bins)

cnnpz.plot_stats(stats2, redshift_stats2, imag_stats2, Y_y10_pre, y_pred_ensemble.flatten(), 
           redshift_bins, imag_bins, pretraining_y10['mag_i_lsst'], save_path=save_dir + '/photoz_stats.png')

In [ ]:
# fine-tune the pre-trained model on spec-select data (or load if already fine-tuned)
root = "/pscratch/sd/q/qhang/cnnpz/noisy_Cardinal/models"
save_dir = root + "/y10_finetune_curve_9band_ensemble_CNN_6layers_noshape"
if os.path.exists(os.path.join(save_dir, "norm_params.json")):
    trained_models_finetune = cnnpz.load_ensemble(save_dir=save_dir)
    histories_finetune = None
else:
    trained_models_finetune, histories_finetune = cnnpz.fine_tune_pre_trained_model(
        X_y10_spec, Y_y10_spec, pretrained_models=trained_models, model_root="", nlayers_forzen=4)
    cnnpz.save_ensemble(trained_models_finetune, save_dir=save_dir)

In [ ]:
if histories_finetune is not None:
    cnnpz.plot_ensemble_losses(histories_finetune, ylim=(0, 0.05))

In [ ]:
y_pred_ensemble, y_pred_STD = cnnpz.ensemble_predict(trained_models_finetune, X_test_y10)

In [ ]:
redshift_bins = np.linspace(0,2.5,11)
imag_bins = np.linspace(18, 25.5,11)
stats2, redshift_stats2, imag_stats2 = cnnpz.get_all_stats(Y_test_y10, y_pred_ensemble.flatten(), test_y10['mag_i_lsst'], save=True, saveroot=save_dir + "/stats.pkl", 
                                                     redshift_bins = redshift_bins, imag_bins = imag_bins)

cnnpz.plot_stats(stats2, redshift_stats2, imag_stats2, Y_test_y10, y_pred_ensemble.flatten(), 
           redshift_bins, imag_bins, test_y10['mag_i_lsst'], save_path=save_dir + '/photoz_stats.png')

## Curve vs. traditional (block) representation comparison

Assumes `MODEL_ROOT` already has both sets of trained ensembles for every experiment above: the curve-representation ones trained in this notebook (`*_curve_ensemble_CNN_6layers`) and the earlier block-representation ones (`*_ensemble_CNN_6layers`, no `curve` tag). `cnnpz`'s `transform_data_to_XY`/`convert_data_format` are curve-only on this branch, so the block-representation data prep is reproduced locally below (self-contained, not touching the `cnnpz` package) purely to feed the block models their matching input. Each experiment's two models are evaluated on the same held-out galaxies (each fed its own matching representation) and compared on biweight-scale bias/`sigma_z`/outlier rate.

In [ ]:
# --- Block-based (traditional) representation, reconstructed locally for comparison ---
# reproduces the pre-curve data prep: hard-assign each wavelength bin to whichever single
# band dominates there (no overlap-sharing), 30 bins across the same wavelength range.

def convert_data_format_block(df, lambda_array_cen, filter_blocks, no_detect=np.nan, no_obs=np.inf,
                               no_detect_val=0, no_obs_val=0):
    num_galaxies = len(df)
    num_wavelength_bins = len(lambda_array_cen)

    mags_data = np.zeros((num_galaxies, num_wavelength_bins))
    filter_availability_data = np.zeros((num_galaxies, num_wavelength_bins))

    for b in "ugrizy":
        ind = filter_blocks[b].astype(bool)
        mags_data[:, ind] = df[f"mag_{b}_lsst"].to_numpy()[:, None]
        filter_availability_data[:, ind] = 1

    for b in "JH":
        ind = filter_blocks[b].astype(bool)
        mags_data[:, ind] = df[f"mag_{b}_roman"].to_numpy()[:, None]
        filter_availability_data[:, ind] = 1

    mags_data -= df["mag_i_lsst"].to_numpy()[:, None]  # normalize by i-band mag

    ind_nan = np.isnan(mags_data)  # no detection
    ind_inf = np.isinf(mags_data)  # no observation
    filter_availability_data[ind_inf] = 0
    mags_data[ind_inf] = no_obs_val
    mags_data[ind_nan] = no_detect_val

    wave_labels = np.arange(num_wavelength_bins)
    wave_labels = wave_labels / wave_labels[-1]
    lambda_labels = np.outer(np.ones(num_galaxies), wave_labels)

    return np.stack([mags_data, lambda_labels, filter_availability_data], axis=-1)


def transform_data_to_XY_block(data, lambda_array_cen, filter_blocks, apply_stretch=True, c=0.8, k=20, missingY=False):
    data_transformed = convert_data_format_block(data, lambda_array_cen, filter_blocks)
    Y = data["redshift"] if not missingY else 0
    X = np.copy(data_transformed)
    if apply_stretch:
        X[:, :, 0] = cnnpz.stretch(X[:, :, 0], c=c, k=k)
    return X, Y


def make_incomplete_nir_data_block(data, lambda_array_cen, filter_blocks, frac=0.5, sub_val=np.inf, apply_stretch=False):
    subset = data.sample(frac=frac)
    idx = list(subset.index)
    data_copy = data.copy()
    data_copy.loc[idx, "mag_J_roman"] = sub_val
    data_copy.loc[idx, "mag_H_roman"] = sub_val
    return transform_data_to_XY_block(data_copy, lambda_array_cen, filter_blocks, apply_stretch=apply_stretch)


# reconstruct the 30-bin block grid exactly as the earlier block-representation notebook did
lambda_array = np.linspace(lambda_min, lambda_max, 31)

bin_edges_block = {}
for b in "ugrizyJH":
    bin_edges_block[b] = cnnpz.get_bin_edges(
        lsst_filter_curves[b][:, 0] if b not in "JH" else roman_filter_curves[b][:, 0]
    )

rebinned_filters_block = {}
for b in "ugrizyJH":
    counts = lsst_filter_curves[b][:, 1] if b not in "JH" else roman_filter_curves[b][:, 1]
    rebinned_filters_block[b] = cnnpz.rebin_filter(bin_edges_block[b], counts, lambda_array)

lambda_array_cen = (lambda_array[1:] + lambda_array[:-1]) / 2

filter_blocks = {}
block_bands = "ugrizyJH"
for i, b in enumerate("ugrizyJ"):
    if i > 0:
        filter_blocks[b] = (rebinned_filters_block[b] >= rebinned_filters_block[block_bands[i + 1]]) & (
            rebinned_filters_block[b] > rebinned_filters_block[block_bands[i - 1]]
        )
    else:
        filter_blocks[b] = rebinned_filters_block[b] > rebinned_filters_block[block_bands[i + 1]]
filter_blocks["H"] = (rebinned_filters_block["H"] > rebinned_filters_block["J"]) & (
    rebinned_filters_block["H"] > rebinned_filters_block["y"]
)

In [ ]:
# build the block-representation counterpart of every test/eval set used above
X_test_y1_block, Y_test_y1_block = transform_data_to_XY_block(test_y1, lambda_array_cen, filter_blocks, apply_stretch=False)
X_test_y1_misnir_block, Y_test_y1_misnir_block = make_incomplete_nir_data_block(
    test_y1, lambda_array_cen, filter_blocks, apply_stretch=False
)

X_test_y10_block, Y_test_y10_block = transform_data_to_XY_block(test_y10, lambda_array_cen, filter_blocks, apply_stretch=False)
X_test_y10_misnir_block, Y_test_y10_misnir_block = make_incomplete_nir_data_block(
    test_y10, lambda_array_cen, filter_blocks, apply_stretch=False
)

# pretraining_y10 has already been redshift-cut, sub-sampled, and index-reset above --
# reuse it as-is so the block and curve pre-training evaluations see the same galaxies
X_y10_pre_block, Y_y10_pre_block = transform_data_to_XY_block(pretraining_y10, lambda_array_cen, filter_blocks, apply_stretch=False)

print(X_test_y1_block.shape, X_test_y10_block.shape, X_y10_pre_block.shape)

In [ ]:
redshift_bins = np.linspace(0, 2.5, 11)
imag_bins = np.linspace(18, 25.5, 11)


def evaluate_experiment(name, curve_dir, block_dir, X_curve, Y_curve, imag_curve, X_block, Y_block, imag_block):
    """Load the curve- and block-representation ensembles for one experiment, predict on
    each one's matching held-out set, and compare their biweight bias/sigma_z/outlier stats."""
    curve_models = cnnpz.load_ensemble(save_dir=curve_dir)
    block_models = cnnpz.load_ensemble(save_dir=block_dir)

    y_pred_curve, _ = cnnpz.ensemble_predict(curve_models, X_curve)
    y_pred_block, _ = cnnpz.ensemble_predict(block_models, X_block)

    stats_curve, rs_curve, is_curve = cnnpz.get_all_stats(
        Y_curve, y_pred_curve.flatten(), imag_curve, save=False, redshift_bins=redshift_bins, imag_bins=imag_bins
    )
    stats_block, rs_block, is_block = cnnpz.get_all_stats(
        Y_block, y_pred_block.flatten(), imag_block, save=False, redshift_bins=redshift_bins, imag_bins=imag_bins
    )

    print(f"=== {name} ===")
    print(cnnpz.stats_to_markdown(stats_block, stats_curve, data_title=("block", "curve")))
    cnnpz.compare_binned_stats(redshift_bins, imag_bins, rs_block, is_block, rs_curve, is_curve)
    plt.suptitle(name)
    plt.show()

    # stats tuple is (mean, mean_err, std, outlier_rate, abs_outlier_rate) -- std is biweight sigma_z
    sigma_block, sigma_curve = stats_block[2], stats_curve[2]
    winner = "curve" if sigma_curve < sigma_block else "block"
    return {
        "experiment": name,
        "sigma_z_block": sigma_block,
        "sigma_z_curve": sigma_curve,
        "outlier_rate_block": stats_block[3],
        "outlier_rate_curve": stats_curve[3],
        "winner (lower sigma_z)": winner,
    }

In [ ]:
root = MODEL_ROOT
pretrain_root = "/pscratch/sd/q/qhang/cnnpz/noisy_Cardinal/models"

results = []

results.append(evaluate_experiment(
    "Y1 complete",
    root + "/y1_complete_curve_ensemble_CNN_6layers", root + "/y1_complete_ensemble_CNN_6layers",
    X_test_y1, Y_test_y1, test_y1['mag_i_lsst'],
    X_test_y1_block, Y_test_y1_block, test_y1['mag_i_lsst'],
))

results.append(evaluate_experiment(
    "Y1 NIR-dropout",
    root + "/y1_misnir_curve_ensemble_CNN_6layers", root + "/y1_misnir_ensemble_CNN_6layers",
    X_test_y1_misnir, Y_test_y1_misnir, test_y1['mag_i_lsst'],
    X_test_y1_misnir_block, Y_test_y1_misnir_block, test_y1['mag_i_lsst'],
))

results.append(evaluate_experiment(
    "Y10 complete",
    root + "/y10_complete_curve_ensemble_CNN_6layers", root + "/y10_complete_ensemble_CNN_6layers",
    X_test_y10, Y_test_y10, test_y10['mag_i_lsst'],
    X_test_y10_block, Y_test_y10_block, test_y10['mag_i_lsst'],
))

results.append(evaluate_experiment(
    "Y10 NIR-dropout",
    root + "/y10_misnir_curve_ensemble_CNN_6layers", root + "/y10_misnir_ensemble_CNN_6layers",
    X_test_y10_misnir, Y_test_y10_misnir, test_y10['mag_i_lsst'],
    X_test_y10_misnir_block, Y_test_y10_misnir_block, test_y10['mag_i_lsst'],
))

results.append(evaluate_experiment(
    "Y1 spec-select (eval on full Y1 test)",
    root + "/y1_specsel_curve_ensemble_CNN_6layers", root + "/y1_specsel_ensemble_CNN_6layers",
    X_test_y1, Y_test_y1, test_y1['mag_i_lsst'],
    X_test_y1_block, Y_test_y1_block, test_y1['mag_i_lsst'],
))

results.append(evaluate_experiment(
    "Y10 spec-select (eval on full Y10 test)",
    root + "/y10_specsel_curve_ensemble_CNN_6layers", root + "/y10_specsel_ensemble_CNN_6layers",
    X_test_y10, Y_test_y10, test_y10['mag_i_lsst'],
    X_test_y10_block, Y_test_y10_block, test_y10['mag_i_lsst'],
))

results.append(evaluate_experiment(
    "Y10 pre-training (eval on pre-training set)",
    pretrain_root + "/y10_pretrain_curve_ensemble_CNN_6layers", pretrain_root + "/y10_pretrain_ensemble_CNN_6layers",
    X_y10_pre, Y_y10_pre, pretraining_y10['mag_i_lsst'],
    X_y10_pre_block, Y_y10_pre_block, pretraining_y10['mag_i_lsst'],
))

results.append(evaluate_experiment(
    "Y10 fine-tune (eval on Y10 test)",
    pretrain_root + "/y10_finetune_curve_ensemble_CNN_6layers", pretrain_root + "/y10_finetune_ensemble_CNN_6layers",
    X_test_y10, Y_test_y10, test_y10['mag_i_lsst'],
    X_test_y10_block, Y_test_y10_block, test_y10['mag_i_lsst'],
))

In [ ]:
summary = pd.DataFrame(results)
n_curve_wins = (summary["winner (lower sigma_z)"] == "curve").sum()
print(summary.to_string(index=False))
print(f"\ncurve representation wins {n_curve_wins}/{len(summary)} experiments (lower biweight sigma_z)")
summary